# Lectura de paquetes y data

In [1]:
import warnings
import os
import time
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna

# Agrega todo el directorio padre al path
sys.path.append(os.path.abspath(".."))
from src.utils_ml import ml_training_utils as ml_utils
from src.utils_ml import ml_feature_engineering as fe_utils

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

/opt/homebrew/Caskroom/miniconda/base/envs/maestria/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Utilizamos paths relativos para la lectura de la data

In [2]:
filename = "ml_pipeline_p_sku.ipynb"  # nombre del archivo actual
print(f"Current absolute path: {os.getcwd()}\n")

# Especificamos la ruta del directorio actual y los directorios de datos y salida
ACTUAL_DIR = os.path.dirname(os.path.abspath(filename))
BASE_DIR = os.path.dirname(ACTUAL_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

Current absolute path: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/notebooks

BASE_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi
DATA_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/data
OUTPUT_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/data/output


In [3]:
# Cargar el archivo de Excel
file_path = os.path.join(DATA_DIR, "data_demanda.xlsx")
df_base = pd.read_excel(file_path, sheet_name="data")
df_base = df_base.drop("Cliente", axis=1)

df_base.shape

(6859, 11)

In [4]:
df_base.head(5)

,Fe.prefer.entrega,Day_of_the_Week,Pedidos,Sell_In,Sell_in_on_time,CEDIS,SKU,Mes_Año,Sell_in_rezago,porcentaje_on_time,Subcategoria
0,2024-07-05,Friday,16200,16200,16200,7,SKU7,72024,0,1.0,BEBIDAS DE YOGUR
1,2024-07-05,Friday,19800,19800,19800,6,SKU5,72024,0,1.0,BEBIDAS DE YOGUR
2,2024-07-05,Friday,4950,4950,4950,4,SKU6,72024,0,1.0,BEBIDAS DE YOGUR
3,2024-07-05,Friday,8100,8100,8100,6,SKU4,72024,0,1.0,BEBIDAS DE YOGUR
4,2024-07-05,Friday,18600,18600,18600,7,SKU3,72024,0,1.0,BEBIDAS DE YOGUR


In [5]:
# Filtrar los datos relevantes para este analisis

df = (
    df_base[["Fe.prefer.entrega", "SKU", "Pedidos"]]
    .copy()
    .rename(
        columns={
            "Fe.prefer.entrega": "Fecha",
        }
    )
)
df["Pedidos"] = pd.to_numeric(df["Pedidos"], errors="coerce")

In [6]:
df

,Fecha,SKU,Pedidos
0,2024-07-05,SKU7,16200
1,2024-07-05,SKU5,19800
2,2024-07-05,SKU6,4950
3,2024-07-05,SKU4,8100
4,2024-07-05,SKU3,18600
...,...,...,...
6854,2025-04-11,SKU14,2400
6855,2025-04-11,SKU16,5550
6856,2025-04-11,SKU8,600
6857,2025-04-11,SKU22,600


# Preparación de la data

In [7]:
### Primero, nos aseguramos de que se cuente un dato por SKU por dia
# -------

# rango completo de fechas desde la más antigua hasta la más reciente
fecha_min = df["Fecha"].min()
fecha_max = df["Fecha"].max()
rango_fechas = pd.date_range(start=fecha_min, end=fecha_max, freq="D")

# Obtenemos todos los SKUs únicos
skus = df["SKU"].unique()

# DataFrame con todas las combinaciones de SKU y fecha
combinaciones_completas = pd.MultiIndex.from_product(
    [rango_fechas, skus], names=["Fecha", "SKU"]
).to_frame(index=False)

# Unir con el dataframe original para rellenar con ceros donde falten datos
df_completo = combinaciones_completas.merge(df, on=["Fecha", "SKU"], how="left")

# Rellenar valores faltantes de pedidos con 0
df_completo["Pedidos"] = df_completo["Pedidos"].fillna(0).astype(int)

# Ordenar por SKU y Fecha (opcional)
df_completo = df_completo.sort_values(["SKU", "Fecha"]).reset_index(drop=True)

df = df_completo.copy()

In [8]:
# Modificar nombre de columnas
df.columns = df.columns.str.replace(".", "_", regex=False).str.lower()

In [9]:
df.shape

(7306, 3)

# EDA

In [10]:
df.isna().sum()

fecha      0
sku        0
pedidos    0
dtype: int64

In [11]:
# porcentaje de ceros por sku
porcentaje_ceros = (
    df.groupby("sku")["pedidos"]
    .apply(lambda x: (x == 0).mean() * 100)
    .reset_index(name="prct_ceros")
    .round(2)
)

# promedio, mediana y desviacion estandar por sku excluyendo ceros
df_temp = df[df["pedidos"] > 0].copy()
promedio = df_temp.groupby("sku")["pedidos"].mean().reset_index(name="Promedio").round()
mediana = df_temp.groupby("sku")["pedidos"].median().reset_index(name="Mediana").round()
desviacion = (
    df_temp.groupby("sku")["pedidos"].std().reset_index(name="Desviacion").round()
)
maximo = df_temp.groupby("sku")["pedidos"].max().reset_index(name="Maximo").round()

# Porcentaje de valores outliers por SKU excluyendo ceros
porcentaje_outliers = (
    df_temp.groupby("sku")["pedidos"]
    .apply(fe_utils.calcular_outliers_porcentaje)
    .reset_index(name="prct_outliers")
    .round(2)
)

# Unir las tablas
tabla_total = pd.merge(porcentaje_ceros, porcentaje_outliers, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, promedio, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, mediana, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, desviacion, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, maximo, on="sku", how="outer")
tabla_total.sort_values(by="prct_ceros", ascending=False)

,sku,prct_ceros,prct_outliers,Promedio,Mediana,Desviacion,Maximo
6,SKU15,16.73,7.26,1001.0,840.0,647.0,3960
7,SKU16,14.23,2.07,4627.0,4725.0,1171.0,7500
0,SKU1,12.46,1.63,17842.0,15000.0,13843.0,84600
24,SKU8,11.74,5.24,1065.0,900.0,892.0,5160
4,SKU13,11.03,4.40,502.0,420.0,284.0,1500
3,SKU12,10.32,3.97,463.0,420.0,286.0,1800
8,SKU17,7.83,2.70,11489.0,9060.0,7480.0,52200
25,SKU9,7.83,8.49,801.0,600.0,640.0,6000
5,SKU14,7.83,2.70,2705.0,1920.0,2084.0,15240
10,SKU19,6.05,3.41,3104.0,2640.0,1955.0,10800


In [12]:
sku = "SKU5"
print(f"Analizando el SKU: {sku}")

Analizando el SKU: SKU5


In [13]:
df_sku = df[df["sku"] == sku].copy()
df_sku = df_sku.drop("sku", axis=1)

# graficamos la serie de tiempo del SKU seleccionado usando plotly
fig = px.line(
    df_sku,
    x="fecha",
    y="pedidos",
    title=f"Serie de tiempo de Pedidos para {sku}:",
)
fig.update_layout(
    xaxis_title="Fecha",
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_traces(line=dict(color="blue", width=2))
fig.show()


# Feature engineering

## Variables temporales

In [14]:
df = fe_utils.create_temporal_features(df, "fecha")
df.shape, df.columns

((7306, 10),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena',
        'is_vacation'],
       dtype='object'))

## Variables tipo lag

In [15]:
df = fe_utils.create_lag_features(df, "pedidos", "sku", "fecha", 14, 4)
df.shape, df.columns

((7306, 28),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3',
        'pedidos_weekday_lag_4'],
       dtype='object'))

## Variables tipo promedio moviles

In [16]:
df = fe_utils.create_rolling_features(df, "pedidos", "sku", "fecha")
df.shape, df.columns

((7306, 32),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3',
        'pedidos_weekday_lag_4', 'pedidos_rolling_2', 'pedidos_rolling_7',
        'pedidos_prev_week_avg', 'pedidos_dow_avg_2wks'],
       dtype='object'))

## Variables lags de STL

## Aplanamiento de outliers en demanda

# Modelling 

## Evaluación modelos XGBoost

## Evaluación modelos Random Forest

## Evaluación modelos Elastic Net

## Selección mejor modelo y ajuste final

# Predicción y graficas 